# Essential Imports and Configurations

In [1]:
import pandas as pd
import sqlalchemy as sqla
import os
import re

from datetime import datetime
from pathlib import Path
from sqlalchemy import create_engine, text

from dotenv import load_dotenv
from typing import Dict, Optional

In [2]:
load_dotenv()

user: str = os.getenv("DB_USER")
password: str = os.getenv("DB_PASSWORD")
host: str = os.getenv("DB_HOST")
port: str = os.getenv("PORT")
database: str = os.getenv("DB_NAME")

In [3]:
pd.set_option('display.max_rows', None)      # Show all rows
pd.set_option('display.max_columns', None)   # Show all columns
pd.set_option('display.width', 1000)         # Prevents wrapping

In [4]:
def get_connection() -> sqla.Engine:
    engine: sqla.Engine = create_engine(
        f"postgresql://{user}:{password}@{host}:{port}/{database}"
    )
    return engine

try:
    engine: sqla.Engine = get_connection()
    print(f"Connection to the {host} for user {user} created successfully.")

except Exception as ex:
    print("Connection could not be made due to the following error:\n", ex)

Connection to the localhost for user postgres created successfully.


In [5]:
def ingest_data() -> None:
    """
    Populates the Cyber Databoard database with the CSV files data.
    """
    engine: sqla.Engine = get_connection()
    parent: str = (Path.cwd())
    path: Path = Path(f"{parent}/data/")
    day_regex: str = r"[\w\-.]+(?=-[Ww]orkingHours)"
    type_regex: str = r"(?<=-[Ww]orkingHours-)[\w\-.]+(?=.pcap)"

    for file_name in path.iterdir():
        with open(file_name, mode='r') as file:
            datafile: pd.DataFrame = pd.read_csv(file)
            day_match: re.Match = re.search(day_regex, str(file_name.name))
            type_match: re.Match = re.search(type_regex, str(file_name))
            if day_match:
                day_match: str = day_match.group().replace("_", " ")
            if type_match:
                type_match: str = type_match.group().replace("_", " ")
            else:
                type_match: str = "Working_Hours"

            table_name: str = f"{day_match}_{type_match}".replace(" ", "_")
            
            datafile.to_sql(
                f"{table_name}",
                engine, 
                if_exists="replace",
                index=False
            )


# Data Analysis

In [ ]:
def query_database(query: str, para_bindings: Optional[Dict[str, str | int]] = None) -> pd.DataFrame:
    with engine.connect() as connection:
        query: sqla.Text = text(query)

        result: sqla.CursorResult = connection.execute(query, para_bindings)
        df: pd.DataFrame = pd.DataFrame(result)
        
        return df

# Friday_DDOS Analysis

This dataset seems to contain the data from DDOS attacks that occured during Friday Afternoon.

In [7]:
Friday_DDOS: pd.DataFrame = query_database('SELECT * FROM "Friday_Afternoon_DDos"')

In [8]:
Friday_DDOS.info()

<class 'pandas.DataFrame'>
RangeIndex: 225745 entries, 0 to 225744
Data columns (total 85 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Flow ID                       225745 non-null  str    
 1    Source IP                    225745 non-null  str    
 2    Source Port                  225745 non-null  int64  
 3    Destination IP               225745 non-null  str    
 4    Destination Port             225745 non-null  int64  
 5    Protocol                     225745 non-null  int64  
 6    Timestamp                    225745 non-null  str    
 7    Flow Duration                225745 non-null  int64  
 8    Total Fwd Packets            225745 non-null  int64  
 9    Total Backward Packets       225745 non-null  int64  
 10  Total Length of Fwd Packets   225745 non-null  int64  
 11   Total Length of Bwd Packets  225745 non-null  int64  
 12   Fwd Packet Length Max        225745 non-null  int64  


In [9]:
Friday_DDOS.describe()

C:\Users\emera\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\core\nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
C:\Users\emera\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\core\nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,225745.000000,225745.00000,225745.000000,2.257450e+05,225745.000000,225745.000000,225745.000000,2.257450e+05,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,2.257410e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,225745.000000,225745.0,225745.0,225745.0,225745.000000,225745.000000,2.257450e+05,2.257450e+05,225745.000000,225745.000000,225745.000000,225745.000000,2.257450e+05,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,225745.0,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,225745.0,225745.0,225745.0,225745.0,225745.0,225745.0,225745.000000,225745.000000,225745.000000,2.257450e+05,225745.000000,225745.000000,225745.000000,225745.000000,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05
mean,38257.568402,8879.61946,7.600288,1.624165e+07,4.874916,4.572775,939.463346,5.960477e+03,538.535693,27.882221,164.826715,214.907242,2735.585147,16.718776,890.536849,1230.172938,inf,inf,1.580587e+06,4.248569e+06,1.348977e+07,2.811855e+04,1.539652e+07,2.540610e+06,5.195207e+06,1.299434e+07,2.073698e+05,6.564701e+06,9.476322e+05,1.610306e+06,4.567514e+06,2.257817e+05,0.033223,0.0,0.0,0.0,111.522718,106.789023,1.261508e+04,1.641693e+03,8.072595,3226.045339,515.002137,1085.593207,2.789906e+06,0.002671,0.033223,0.000120,0.351162,0.504463,0.140752,0.0,0.000120,1.005821,574.568843,164.826715,890.536849,111.522718,0.0,0.0,0.0,0.0,0.0,0.0,4.874916,939.463346,4.572775,5.960477e+03,4247.436922,601.048635,3.311497,21.482753,1.848261e+05,1.293436e+04,2.080849e+05,1.776201e+05,1.032214e+07,3.611943e+06,1.287813e+07,7.755355e+06
std,23057.302075,19754.64740,3.881586,3.152437e+07,15.422874,21.755356,3249.403484,3.921834e+04,1864.128991,163.324159,504.892965,797.411073,3705.123460,50.480568,1120.324921,1733.201267,NaN,NaN,2.701596e+06,7.622819e+06,2.670172e+07,7.598100e+05,3.160826e+07,5.934694e+06,1.078635e+07,2.748870e+07,3.795228e+06,2.198455e+07,4.586374e+06,5.475778e+06,1.617865e+07,4.019290e+06,0.179220,0.0,0.0,0.0,375.790727,511.765795,1.106701e+05,1.989593e+04,15.767713,3813.134850,559.064495,1269.558714,4.115941e+06,0.051614,0.179220,0.010936,0.477334,0.499981,0.347766,0.0,0.010936,1.430781,626.096202,504.892965,1120.324921,375.790727,0.0,0.0,0.0,0.0,0.0,0.0,15.422874,3249.403484,21.755356,3.921834e+04,8037.781019,4319.720339,12.270018,4.166799,7.979250e+05,2.102737e+05,9.002350e+05,7.842602e+05,2.185303e+0

In [10]:
Friday_DDOS.isnull().any()

# Only Flow Bytes have null values

Flow ID                         False
 Source IP                      False
 Source Port                    False
 Destination IP                 False
 Destination Port               False
 Protocol                       False
 Timestamp                      False
 Flow Duration                  False
 Total Fwd Packets              False
 Total Backward Packets         False
Total Length of Fwd Packets     False
 Total Length of Bwd Packets    False
 Fwd Packet Length Max          False
 Fwd Packet Length Min          False
 Fwd Packet Length Mean         False
 Fwd Packet Length Std          False
Bwd Packet Length Max           False
 Bwd Packet Length Min          False
 Bwd Packet Length Mean         False
 Bwd Packet Length Std          False
Flow Bytes/s                     True
 Flow Packets/s                 False
 Flow IAT Mean                  False
 Flow IAT Std                   False
 Flow IAT Max                   False
 Flow IAT Min                   False
Fwd IAT Tota

In [11]:
Friday_DDOS['Flow Bytes/s'].isnull().sum()

np.int64(4)

In [12]:
# Since there are so little missing values, we can just drop them

print(f"Before removing missing values: {Friday_DDOS.shape}")
Friday_DDOS.dropna(inplace=True)
print(f"After removing missing values: {Friday_DDOS.shape}")

Before removing missing values: (225745, 85)
After removing missing values: (225741, 85)


In [13]:
str_col  = Friday_DDOS.select_dtypes(include='str')
print(str_col.columns)

# We are going to have to handle all these non-string columns expect Label for ML 
# Timestamp can be converted to date format but everything else will need to be encoded as values represent IP addresses.

Index(['Flow ID', ' Source IP', ' Destination IP', ' Timestamp', ' Label'], dtype='str')


In [14]:
Friday_DDOS[' Timestamp'] = pd.to_datetime(Friday_DDOS[' Timestamp'], format="%d/%m/%Y %H:%M")

Friday_DDOS.info()

<class 'pandas.DataFrame'>
Index: 225741 entries, 0 to 225744
Data columns (total 85 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   Flow ID                       225741 non-null  str           
 1    Source IP                    225741 non-null  str           
 2    Source Port                  225741 non-null  int64         
 3    Destination IP               225741 non-null  str           
 4    Destination Port             225741 non-null  int64         
 5    Protocol                     225741 non-null  int64         
 6    Timestamp                    225741 non-null  datetime64[us]
 7    Flow Duration                225741 non-null  int64         
 8    Total Fwd Packets            225741 non-null  int64         
 9    Total Backward Packets       225741 non-null  int64         
 10  Total Length of Fwd Packets   225741 non-null  int64         
 11   Total Length of Bwd Packets 

1. What are the most common ports used here?

In [15]:
ports = query_database('SELECT " Destination Port", " Label", COUNT(*) AS total FROM "Friday_Afternoon_DDos" GROUP BY  " Destination Port", " Label" ORDER BY COUNT(*) DESC')

In [16]:
ports.head(15)

# Here we can see that all DDoS attacks use port 80

,Destination Port,Label,total
0,80,DDoS,128024
1,53,BENIGN,31950
2,443,BENIGN,13485
3,80,BENIGN,8927
4,8080,BENIGN,510
5,123,BENIGN,362
6,22,BENIGN,342
7,137,BENIGN,274
8,389,BENIGN,261
9,88,BENIGN,173


### 2. What does the average packet size look like?
Answer: Results vary drastically but we can see that most packets are around 7.5. 


In [1]:
packet_size = query_database('''SELECT " Average Packet Size", " Label", COUNT(*) AS total FROM "Friday_Afternoon_DDos" GROUP BY  " Average Packet Size", " Label" ORDER BY total DESC''')
packet_size.head(15)

NameError: name 'query_database' is not defined

In [18]:
#Thought packet size might indicate an attack. 
#Sorting by Packet size instead
packet_size = query_database('''SELECT " Average Packet Size", " Label", COUNT(*) AS total FROM "Friday_Afternoon_DDos" GROUP BY  " Average Packet Size", " Label" ORDER BY " Average Packet Size" DESC''')
packet_size.head(15)

#Turns out size is not an indication of malicious packets after all.

,Average Packet Size,Label,total
0,2528.000000,BENIGN,1
1,2357.000000,BENIGN,1
2,2324.200000,DDoS,5
3,2324.200000,BENIGN,1
4,2068.000000,BENIGN,30
5,1937.833333,DDoS,244
6,1937.833333,BENIGN,24
7,1936.833333,BENIGN,30
8,1936.833333,DDoS,233
9,1661.857143,BENIGN,581


### 3. What are the most common protocols?
Answer:
1. Protocol 6 = TCP 
2. Protocol 17 = UDP 
3. Protocol 0 = N/A

In [19]:
protocols = query_database('SELECT " Protocol", " Label", COUNT(*) AS total FROM "Friday_Afternoon_DDos" GROUP BY  " Protocol", " Label" ORDER BY total DESC')
protocols
#DDoS attacks seem to use TCP

,Protocol,Label,total
0,6,DDoS,128027
1,6,BENIGN,64793
2,17,BENIGN,32871
3,0,BENIGN,54


### 4. How many DDoS attacks where there total, Friday Afternoon?
Answer: 128,027

In [20]:
num_attacks = query_database('SELECT " Label", COUNT(*) AS total FROM "Friday_Afternoon_DDos" GROUP BY " Label" ORDER BY total DESC')
num_attacks

,Label,total
0,DDoS,128027
1,BENIGN,97718


In [25]:
Friday_DDOS[' Label'].unique()

<ArrowStringArray>
['BENIGN', 'DDoS']
Length: 2, dtype: str

### 5. What time did each attack take place at?
Answer: The attack seemed to occur from 3:57pm to 4:16pm

In [30]:
times = query_database('SELECT " Timestamp", COUNT(*) AS total FROM "Friday_Afternoon_DDos" WHERE " Label" =  \'DDoS\'  GROUP BY " Timestamp" ORDER BY total DESC')
times

,Timestamp,total
0,7/7/2017 3:57,8244
1,7/7/2017 4:01,6977
2,7/7/2017 4:02,6889
3,7/7/2017 4:03,6778
4,7/7/2017 3:59,6607
5,7/7/2017 3:58,6576
6,7/7/2017 4:12,6552
7,7/7/2017 4:00,6533
8,7/7/2017 4:09,6466
9,7/7/2017 4:14,6435


# Friday_PortScan Analysis

In [32]:
Friday_PS: pd.DataFrame = query_database('SELECT * FROM "Friday_Afternoon_PortScan"')

In [ ]:
Friday_PS.info()

#Exact same as the Friday_DDoS information, we only have to address the date time

<class 'pandas.DataFrame'>
RangeIndex: 286467 entries, 0 to 286466
Data columns (total 85 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Flow ID                       286467 non-null  str    
 1    Source IP                    286467 non-null  str    
 2    Source Port                  286467 non-null  int64  
 3    Destination IP               286467 non-null  str    
 4    Destination Port             286467 non-null  int64  
 5    Protocol                     286467 non-null  int64  
 6    Timestamp                    286467 non-null  str    
 7    Flow Duration                286467 non-null  int64  
 8    Total Fwd Packets            286467 non-null  int64  
 9    Total Backward Packets       286467 non-null  int64  
 10  Total Length of Fwd Packets   286467 non-null  int64  
 11   Total Length of Bwd Packets  286467 non-null  int64  
 12   Fwd Packet Length Max        286467 non-null  int64  


In [34]:
Friday_PS.describe()

C:\Users\emera\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\core\nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
C:\Users\emera\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\core\nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,286467.000000,286467.000000,286467.000000,2.864670e+05,286467.000000,286467.000000,286467.000000,2.864670e+05,286467.000000,286467.000000,286467.000000,286467.000000,286467.000000,286467.000000,286467.000000,286467.000000,2.864520e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,286467.000000,286467.0,286467.0,286467.0,286467.000000,286467.000000,2.864670e+05,2.864670e+05,286467.000000,286467.000000,286467.000000,286467.000000,2.864670e+05,286467.000000,286467.000000,286467.000000,286467.000000,286467.000000,286467.000000,286467.0,286467.000000,286467.000000,286467.000000,286467.000000,286467.000000,286467.000000,286467.0,286467.0,286467.0,286467.0,286467.0,286467.0,286467.000000,286467.000000,286467.000000,2.864670e+05,286467.000000,286467.000000,286467.000000,286467.000000,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05,2.864670e+05
mean,43932.432563,8044.876324,8.306000,5.379331e+06,3.473283,3.520500,233.407667,2.707247e+03,81.033369,10.352013,24.138688,24.590994,185.649851,27.478530,76.473013,55.742307,inf,inf,3.435661e+05,6.690304e+05,1.821288e+06,1.152034e+04,5.255228e+06,7.790948e+05,5.086644e+05,1.773447e+06,4.483619e+05,4.968463e+06,7.972503e+05,3.961929e+05,1.569683e+06,5.053646e+05,0.020994,0.0,0.0,0.0,91.861527,87.156077,4.140927e+04,2.110817e+04,10.124908,206.783141,47.863395,61.756216,3.557227e+04,0.008692,0.020994,0.000073,0.660694,0.124116,0.044672,0.0,0.000073,0.850625,55.157954,24.138688,76.473013,91.861527,0.0,0.0,0.0,0.0,0.0,0.0,3.473283,233.407667,3.520500,2.707247e+03,11145.127076,1155.099977,1.728618,29.075328,3.409213e+04,2.294987e+04,7.785722e+04,2.171380e+04,1.602973e+06,6.120390e+04,1.647188e+06,1.536286e+06
std,17713.904328,15378.583442,4.481562,2.192364e+07,19.515131,28.288916,1865.523600,5.097783e+04,327.768035,24.237795,78.381542,117.218043,585.720606,54.209647,204.152865,196.643780,NaN,NaN,2.124999e+06,3.830433e+06,9.034294e+06,3.800536e+05,2.179685e+07,5.979401e+06,2.575708e+06,9.019385e+06,5.776419e+06,2.125371e+07,6.242734e+06,2.371670e+06,8.712596e+06,6.042689e+06,0.143363,0.0,0.0,0.0,503.560131,698.237440,1.601312e+05,6.246254e+04,18.768649,636.614018,122.031372,178.209296,1.631016e+05,0.092826,0.143363,0.008562,0.473475,0.329714,0.206583,0.0,0.008562,0.450208,129.794071,78.381542,204.152865,503.560131,0.0,0.0,0.0,0.0,0.0,0.0,19.515131,1865.523600,28.288916,5.097783e+04,14274.278654,6645.329412,14.897361,8.014607,4.659319e+05,2.623049e+05,7.342606e+05,4.223145e+05,8.682334e+06,1.217016e+06,8.882

In [38]:
Friday_PS.isnull().any()

#Just Flow Bytes again

Flow ID                         False
 Source IP                      False
 Source Port                    False
 Destination IP                 False
 Destination Port               False
 Protocol                       False
 Timestamp                      False
 Flow Duration                  False
 Total Fwd Packets              False
 Total Backward Packets         False
Total Length of Fwd Packets     False
 Total Length of Bwd Packets    False
 Fwd Packet Length Max          False
 Fwd Packet Length Min          False
 Fwd Packet Length Mean         False
 Fwd Packet Length Std          False
Bwd Packet Length Max           False
 Bwd Packet Length Min          False
 Bwd Packet Length Mean         False
 Bwd Packet Length Std          False
Flow Bytes/s                     True
 Flow Packets/s                 False
 Flow IAT Mean                  False
 Flow IAT Std                   False
 Flow IAT Max                   False
 Flow IAT Min                   False
Fwd IAT Tota

In [39]:
# Since there are so little missing values, we can just drop them again

print(f"Before removing missing values: {Friday_PS.shape}")
Friday_PS.dropna(inplace=True)
print(f"After removing missing values: {Friday_PS.shape}")

Before removing missing values: (286467, 85)
After removing missing values: (286452, 85)


In [40]:
Friday_PS[' Timestamp'] = pd.to_datetime(Friday_PS[' Timestamp'], format="%d/%m/%Y %H:%M")

Friday_PS.info()

<class 'pandas.DataFrame'>
Index: 286452 entries, 0 to 286466
Data columns (total 85 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   Flow ID                       286452 non-null  str           
 1    Source IP                    286452 non-null  str           
 2    Source Port                  286452 non-null  int64         
 3    Destination IP               286452 non-null  str           
 4    Destination Port             286452 non-null  int64         
 5    Protocol                     286452 non-null  int64         
 6    Timestamp                    286452 non-null  datetime64[us]
 7    Flow Duration                286452 non-null  int64         
 8    Total Fwd Packets            286452 non-null  int64         
 9    Total Backward Packets       286452 non-null  int64         
 10  Total Length of Fwd Packets   286452 non-null  int64         
 11   Total Length of Bwd Packets 